# Configuración de LlamaIndex

In [10]:
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

llm_model = OpenAI(
    model="gpt-4o-mini",
    api_key=api_key
)

llm_embeddings = OpenAIEmbedding(
    model="text-embedding-3-small",
    api_key=api_key
)

Settings.llm = llm_model
Settings.embed_model = llm_embeddings

# RAG (con pocas líneas)

In [5]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader('seguros').load_data()

In [7]:
len(documents)

17

In [11]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_documents(documents)

In [12]:
query_engine = index.as_query_engine()

In [13]:
response = query_engine.query("¿Cual es la protección jurídica del seguro de hogar?")

In [15]:
response.response

'La protección jurídica del seguro de hogar está diseñada para ofrecer asesoramiento telefónico ilimitado y defensa en reclamaciones de daños que afecten a la vivienda y a los objetos asegurados. Esta cobertura es especialmente útil si se tiene una vivienda alquilada, cedida a terceros o una segunda vivienda. El gasto máximo cubierto es de 4.000 € en un tipo de seguro, y de 8.000 € en otro, existiendo un importe mínimo reclamado para poder acceder a la cobertura.'

In [20]:
len(response.source_nodes)

2

In [21]:
response.metadata

{'97233dc3-b5a9-4a87-bfab-ed33586ae1af': {'page_label': '5',
  'file_name': 'AF-SL-TablasGarantiasHogar-Completo-Modalidad14-v11-CAS.pdf',
  'file_path': 'C:\\Users\\efclprd\\Desktop\\nlp\\seguros\\AF-SL-TablasGarantiasHogar-Completo-Modalidad14-v11-CAS.pdf',
  'file_type': 'application/pdf',
  'file_size': 96118,
  'creation_date': '2026-03-31',
  'last_modified_date': '2025-10-11'},
 '938a3f22-7e54-4afc-b7b6-fc8b19b3d1a5': {'page_label': '6',
  'file_name': 'AF-SL-TablasGarantiasHogar-Premium-Modalidad15-v13-CAS.pdf',
  'file_path': 'C:\\Users\\efclprd\\Desktop\\nlp\\seguros\\AF-SL-TablasGarantiasHogar-Premium-Modalidad15-v13-CAS.pdf',
  'file_type': 'application/pdf',
  'file_size': 101016,
  'creation_date': '2026-03-31',
  'last_modified_date': '2025-10-11'}}

# Evaluation (LLM-as-a-judge)

In [23]:
from openai import OpenAI
client = OpenAI(api_key=api_key)

In [25]:
gt_questions = [
"tengo alguna garantía que cubra los humos?",
"qué seguros de hogar cubren las placas solares?",
"la vitrocerámica está incluida en alguna modalidad del seguro?",
]

In [26]:
gt_answers = [
"Sí, las modalidades de los seguros Hogar Completo, Hogar Eficaz y Hogar Premium incluyen la cobertura de daños ocasionados por humo, ya sea por escapes repentinos en cocinas, sistemas de calefacción u otros aparatos eléctricos.",
"La modalidad Hogar Completo cubre el robo de placas solares en cubiertas de viviendas unifamiliares hasta 3.000 € por incidente. En la modalidad Hogar Premium, la cobertura para el robo de placas solares se amplía hasta 5.000 € por incidente.",
"Sí, las modalidades Hogar Completo y Hogar Premium cubren la rotura accidental de elementos vitrocerámicos de cocina, incluyendo la reposición de la placa o cristal."
]

In [27]:
# rag answers

rag_answers = []
for q in gt_questions:
    rr = query_engine.query(q)
    rag_answers.append(rr.response)
    print(rr.response)
    print('*'*50)

Sí, hay cobertura para los daños que ocasionen los humos, especialmente por escapes repentinos en cocinas, sistemas de calefacción y otros aparatos eléctricos. Esto incluye, por ejemplo, la limpieza de paredes ennegrecidas por humo.
**************************************************
Los seguros de hogar que cubren las placas solares indemnizan en caso de robo de las mismas si están ubicadas en la cubierta de una vivienda unifamiliar, adosado o pareado. La cobertura puede alcanzar hasta 5.000 € por incidente y anualidad de seguro.
**************************************************
Sí, la reposición de la rotura de la placa o cristal de la vitrocerámica está cubierta en el seguro, en caso de una rotura accidental.
**************************************************


In [34]:
def evaluate_question(ground_truth, llm_response):
    
    grading_prompt = f"""
    Evalúa la calidad de la respuesta generada por el modelo LLM en comparación con la respuesta de referencia (ground truth). \
    A continuación se presentan los criterios para la evaluación: 
    
    - **Score 1:** La respuesta del LLM es irrelevante o incorrecta en comparación con la verdad de referencia. 
    - **Score 2:** La respuesta del LLM es algo relacionada pero tiene errores significativos o está incompleta \
    en comparación con la verdad de referencia. 
    - **Score 3:** La respuesta del LLM es aceptable, pero carece de precisión o detalles importantes en comparación \
    con la verdad de referencia. 
    - **Score 4:** La respuesta del LLM es buena y responde adecuadamente, aunque hay detalles que podrían mejorarse \
    para alinearse más con la verdad de referencia. 
    - **Score 5:** La respuesta del LLM es excelente, clara, completa y tan precisa como la verdad de referencia o \
    incluso aporta detalles adicionales útiles. 
    
    Proporciona un puntaje del 1 al 5 basado en los criterios anteriores. Muestra solo  un numero y nada mas. 
    **Ground truth:** {ground_truth} 
    **Respuesta del LLM:** {llm_response} 
    **Evaluación:** 
    """
    
    completion = client.chat.completions.create(
        model='gpt-4o',
        temperature=0,
        messages=[
            {"role": "system", "content": "Eres un examinador de notas."},
            {"role": "user", "content": grading_prompt}
        ]
    )
    return completion.choices[0].message.content

In [36]:
scores = []
for ga, raga in zip(gt_answers, rag_answers):
    rr = evaluate_question(ga, raga)
    scores.append(rr)

In [37]:
scores

['5', '3', '4']

# Mejoras

In [60]:
import nest_asyncio
nest_asyncio.apply()

In [73]:
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.extractors import QuestionsAnsweredExtractor, KeywordExtractor

text_splitter = SentenceSplitter(chunk_size=300, chunk_overlap=50)
qa = QuestionsAnsweredExtractor(questions=2)
ke = KeywordExtractor(keywords=3)

In [75]:
qa.prompt_template = "Aquí el contexto:\n{context_str}\n\nDada la información de contexto, genera {num_questions} preguntas que puedan ser respondidas con el contenido de este contexto. "

In [77]:
from llama_index.core.ingestion import IngestionPipeline

pipeline = IngestionPipeline(transformations=[
    text_splitter,
    qa,
#     ke,
])

In [78]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader('seguros').load_data()

nodes = pipeline.run(documents=documents)

100%|████████████████████████████████████████████████████████████████████████████████| 105/105 [00:47<00:00,  2.19it/s]


In [79]:
len(nodes)

105

In [80]:
nodes[1]

TextNode(id_='e5feec94-fe36-4648-8408-60017fe2f751', embedding=None, metadata={'page_label': '2', 'file_name': 'AF-SL-TablasGarantiasHogar-Completo-Modalidad14-v11-CAS.pdf', 'file_path': 'C:\\Users\\efclprd\\Desktop\\nlp\\seguros\\AF-SL-TablasGarantiasHogar-Completo-Modalidad14-v11-CAS.pdf', 'file_type': 'application/pdf', 'file_size': 96118, 'creation_date': '2026-03-31', 'last_modified_date': '2025-10-11', 'questions_this_excerpt_can_answer': '1. ¿Qué tipos de daños están cubiertos por las garantías básicas en caso de incendio según el documento?\n2. ¿Qué sucede si hay daños en la vivienda causados por humo debido a un escape repentino en la cocina?'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='5ac8b

In [81]:
# rag

index = VectorStoreIndex(nodes)
query_engine = index.as_query_engine(similarity_top_k=3)

In [82]:
# rag answers

rag_answers = []
for q in gt_questions:
    rr = query_engine.query(q)
    rag_answers.append(rr.response)

In [83]:
# evaluación

scores = []
for ga, raga in zip(gt_answers, rag_answers):
    rr = evaluate_question(ga, raga)
    scores.append(rr)
scores

['5', '3', '5']

In [84]:
import pandas as pd
pd.set_option('display.max_colwidth', None)

df = pd.DataFrame(zip(gt_questions, gt_answers, rag_answers, scores))
df.columns = ['gt_questions', 'gt_answers', 'llm_answers', 'score']
df

,gt_questions,gt_answers,llm_answers,score
0,tengo alguna garantía que cubra los humos?,"Sí, las modalidades de los seguros Hogar Completo, Hogar Eficaz y Hogar Premium incluyen la cobertura de daños ocasionados por humo, ya sea por escapes repentinos en cocinas, sistemas de calefacción u otros aparatos eléctricos.","Sí, hay garantías que cubren los daños ocasionados por el humo. Esto incluye los daños causados por escapes repentinos en cocinas, sistemas de calefacción y otros aparatos eléctricos. Además, se cubren situaciones como la mancha en el techo causada por humo debido a una llamarada de una sartén.",5
1,qué seguros de hogar cubren las placas solares?,"La modalidad Hogar Completo cubre el robo de placas solares en cubiertas de viviendas unifamiliares hasta 3.000 € por incidente. En la modalidad Hogar Premium, la cobertura para el robo de placas solares se amplía hasta 5.000 € por incidente.","Los seguros de hogar que cubren las placas solares son aquellos que ofrecen indemnización en caso de robo de las mismas, específicamente en viviendas unifamiliares, adosadas o pareadas. La cobertura puede variar, con un límite de indemnización de hasta 3,000 € por incidente en una modalidad y hasta 5,000 € en otra.",3
2,la vitrocerámica está incluida en alguna modalidad del seguro?,"Sí, las modalidades Hogar Completo y Hogar Premium cubren la rotura accidental de elementos vitrocerámicos de cocina, incluyendo la reposición de la placa o cristal.","Sí, la vitrocerámica está incluida en las modalidades del seguro, ya que se cubre la reposición de la rotura de la placa o cristal de la vitrocerámica por una rotura accidental.",5


# External Vector Store

In [85]:
index.storage_context.persist('vector_store')

In [88]:
from llama_index.core import StorageContext, load_index_from_storage

sc = StorageContext.from_defaults(persist_dir='vector_store')

index2 = load_index_from_storage(sc)
query_engine2 = index2.as_query_engine()

In [90]:
# query_engine2.query('tengo alguna garantía que cubra humos?')